This is a basic protein-ligand docking workflow, developed as part of a master's thesis project for informed site-directed mutagenesis in proteins without experimentally determined structures.

The workflow achieves the following -
1. Fetches the protein's predicted structure PDB file from the AlphaFold database and removes all non-protein atoms.
2. Fetches a given ligand from PubChem.
3. Prepares structures for action by the docking engine.
4. Sets a search region around the protein for docking.
5. Carries out the default docking regime provided by AutoDock Vina.
6. Converts produced files back to a PDB format and displays it.

Further analyses can be conducted on the obtained structures, such as subjecting them to MD simulations, among other things.

(Note: Do not run on Python Colab versions later than 2026.04.)

The example protein-ligand system in use is the Streptavidin-Biotin system.
The observed binding energy values do not correspond to experimentally obtained values due to a lack of buffer thermodynamics and temperature-entropy considerations.

Install necessary dependencies.

In [ ]:
!pip install -q requests biopython meeko gemmi vina rdkit py3Dmol

In [ ]:
import requests, numpy as np, py3Dmol
from Bio.PDB import PDBParser, PDBIO, Select
from meeko import MoleculePreparation, PDBQTWriterLegacy
from vina import Vina
from rdkit import Chem #Import all requirements

Fetch protein and ligand structures. Clean the protein structure file.

In [ ]:
#Fetch your protein of interest
alphafold_id = "A0A9W9DMP7" #AlphaFold ID for Streptavidin.
with open(alphafold_id + ".pdb", "wb") as protein:
    protein.write(requests.get("https://alphafold.ebi.ac.uk/files/AF-"+ alphafold_id +"-F1-model_v6.pdb").content)

class ProteinOnly(Select): #Select atoms only belonging to the protein.
    def accept_residue(self, residue):
        return residue.id[0] == " " 

protein_of_interest = PDBParser(QUIET=True).get_structure("protein", alphafold_id+".pdb") 
PDB_var = PDBIO()
PDB_var.set_structure(protein_of_interest)
PDB_var.save("my_protein.pdb", ProteinOnly())

#Fetch your ligand of choice. Code can be modidied to bring in multiple ligands as well.
ligand_name = "biotin"
with open("my_ligand.sdf", "wb") as ligand:
    ligand.write(requests.get("https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/"+ ligand_name +"/SDF?record_type=3d").content)

Ligands with many atoms, especially those having a great amount of torsional flexibility, such as peptides greater than a certain length, may not work here.

Convert structures to .pdbqt for Vina.

In [ ]:
!mk_prepare_receptor.py --read_pdb my_protein.pdb -o my_protein -p
!mk_prepare_ligand.py -i my_ligand.sdf -o my_ligand.pdbqt #Converting structures using meeko.

Set up a spatial box for docking.

In [ ]:
atomic_coordinates = []
for atom in protein_of_interest.get_atoms():
    atomic_coordinates.append(atom.coord) #coord provides access for the atomic coordinates held in the parsed structure.
coordinate_array = np.array(atomic_coordinates) #Creates a matrix of all the atomic coordinates.

positional_mean = coordinate_array.mean(axis=0) #Finds the mean of all atomic coordinates.
print("Docking box is centred at :-", positional_mean)
box_dimension = np.clip((coordinate_array.max(axis=0) - coordinate_array.min(axis=0)) + 10, 20, 60)  #Vina's box dimensions in angstroms.
print("Size of the docking box is :-", box_dimension)

Run AutoDock Vina.

In [ ]:
interact = Vina()
interact.set_receptor("my_protein.pdbqt"), interact.set_ligand_from_file("my_ligand.pdbqt")
interact.compute_vina_maps(center=positional_mean.tolist(), box_size=box_dimension.tolist())
interact.dock(exhaustiveness= 10, n_poses=5), interact.write_poses("my_protein_ligand_complex.pdbqt", n_poses=5, overwrite=True)
print(interact.energies(n_poses = 1)) #Run simulation 10 times but store only 5 positions. Print only for the top position.

Running Vina is the most intensive step, as the engine scores the positions of the ligand about the protein's atoms.

Convert ligand positions to the .sdf format.

In [16]:
!mk_export.py my_protein_ligand_complex.pdbqt -s my_ligand_positions.sdf

Convert .sdf to .pdb for viewing with the protein.

In [17]:
sdf_to_pdb = Chem.PDBWriter("my_ligand_positions.pdb")
for count, position in enumerate(Chem.SDMolSupplier("my_ligand_positions.sdf", removeHs=False), start=1):
    if position is None:
        continue
    sdf_to_pdb.write(position)
sdf_to_pdb.close()

View a docked structure.

In [ ]:
mol_view = py3Dmol.view(width=600, height=400)
mol_view.setBackgroundColor("lightgray")

with open("my_protein.pdb") as my_protein:
  mol_view.addModel(my_protein.read(), "pdb")
  mol_view.setStyle({"cartoon": {"color": "blue"}}) #Open your protein in the viewer.

with open("my_ligand_positions.pdb") as my_ligand:
  mol_view.addModel(my_ligand.read(), "pdb")
  mol_view.setStyle({"model": 1}, {"stick": {"colorscheme": "grayCarbon"}}) #Open a ligand position in the viewer.

mol_view.zoomTo(), mol_view.show()

This workflow doesn't incorporate considerations of flexibility of the molecules or buffer conditions, and evaluates using only rigid atomic coordinates for the lowest energy positions.

Consider using pre-equilibrated or relaxed protein structures if not fetching directly from the AlphaFold database.

More dynamic analyses can be done using other options within the AutoDock suite in conjunction with molecular dynamics simulations.